# Concurrency Tests

This notebook compares **pessimistic** and **optimistic** concurrency control for the `library_sim` database.

The test uses multiple Python threads to simulate concurrent database activity and reports metrics such as runtime, successful operations, errors, deadlocks, timeouts, conflicts, and logged transactions.

## 1. Imports

Importing the appropriate packages

In [2]:
import mysql.connector
import threading
import time
from getpass import getpass
import random

## 2. MySQL Connection Settings

Defines the database connection to MySQL. User will be prompted for password after running the cell.

In [3]:
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": getpass("MySQL password: "),
    "database": "library_sim"
}

NUM_THREADS = 50
OPERATIONS_PER_THREAD = 500

## 3. Metrics

Building empty dictionaries for holding our metrics for later analysis.

In [4]:
metrics = {
    "optimistic": {
        "success": 0,
        "errors": 0,
        "deadlocks": 0,
        "timeouts": 0
    },
    "pessimistic": {
        "success": 0,
        "errors": 0,
        "deadlocks": 0,
        "timeouts": 0
    }
}

metric_lock = threading.Lock()

## 4. Database Connection Helper

Function to call the database connection using credentials from above.

In [5]:
def get_connection():
    return mysql.connector.connect(**DB_CONFIG)

## 5. Reset Test Environment

Before testing, clear the log tables and reset the selected book copies.

In [6]:
def reset_test_environment():
    try:
        conn = get_connection()
        cursor = conn.cursor()

        # 1. Clear only the audit and error logs
        cursor.execute("DELETE FROM transaction_log")
        cursor.execute("DELETE FROM conflict_log")
        cursor.execute("DELETE FROM deadlock_log")

        # 2. Reset the mutable state of your book copies (using backticks for reserved keywords)
        cursor.execute("UPDATE book_copy SET `status` = 'Available', `version` = 0")

        # 3. Clear active loans
        cursor.execute("DELETE FROM loan")

        conn.commit()

    except Exception as e:
        print(f"Reset error: {e}")
        try:
            conn.rollback()
        except:
            pass
    finally:
        try:
            cursor.close()
            conn.close()
        except:
            pass

## 6. Pessimistic Worker

Building the `pessimistic_worker` function. Each thread repeatedly calls this function under the pessimistic locking test to simulate multiple concurrent users.

In [7]:
def pessimistic_worker(thread_id, scenario):
    conn = get_connection()
    cursor = conn.cursor()

    max_copies = 2 if scenario == "high_contention" else 15000
    is_high = (scenario == "high_contention")

    for _ in range(OPERATIONS_PER_THREAD):
        copy_id = random.randint(1, max_copies)
        is_write = True if is_high else (random.random() < 0.02)

        try:
            if is_write:
                cursor.callproc("checkout_pessimistic", [copy_id])
                for result in cursor.stored_results():
                    result.fetchall()
                conn.commit()  # Only commit on writes
            else:
                # Pessimistic read with row-level lock
                cursor.execute("SELECT status FROM book_copy WHERE copy_id = %s FOR UPDATE", (copy_id,))
                cursor.fetchall()
                conn.commit()  # Keep transaction closed for consistency

            with metric_lock:
                metrics["pessimistic"]["success"] += 1

        except Exception as e:
            error_text = str(e)
            with metric_lock:
                metrics["pessimistic"]["errors"] += 1
                if "1205" in error_text:
                    metrics["pessimistic"]["timeouts"] += 1
                if "1213" in error_text:
                    metrics["pessimistic"]["deadlocks"] += 1
            try:
                conn.rollback()
            except:
                pass

    cursor.close()
    conn.close()

## 7. Optimistic Worker

Building the `optimistic_worker` function. Each thread repeatedly calls this function under the optimistic locking test to simulate multiple concurrent users.

In [8]:

def optimistic_worker(thread_id, scenario):
    conn = get_connection()
    cursor = conn.cursor()

    max_copies = 2 if scenario == "high_contention" else 15000
    is_high = (scenario == "high_contention")

    for _ in range(OPERATIONS_PER_THREAD):
        copy_id = random.randint(1, max_copies)
        is_write = True if is_high else (random.random() < 0.02)
        success = False
        retries = 0

        while not success and retries < 10:
            try:
                if is_write:
                    cursor.callproc("checkout_optimistic", [copy_id, thread_id, "READ COMMITTED"])
                    for result in cursor.stored_results():
                        result.fetchall()
                    conn.commit()  # Commit only on successful write
                    success = True
                else:
                    # OPTIMISTIC ADVANTAGE: Pure snapshot read with NO commit overhead
                    cursor.execute("SELECT status, version FROM book_copy WHERE copy_id = %s", (copy_id,))
                    cursor.fetchall()
                    # No conn.commit() needed for read-only snapshot!
                    success = True

                with metric_lock:
                    metrics["optimistic"]["success"] += 1

            except Exception as e:
                error_text = str(e)
                if "conflict" in error_text.lower() or retries < 9:
                    retries += 1
                    with metric_lock:
                        metrics["optimistic"]["errors"] += 1
                    continue
                else:
                    with metric_lock:
                        metrics["optimistic"]["errors"] += 1
                    break
            try:
                # Clean up any lingering transaction state if needed
                pass
            except:
                pass

    cursor.close()
    conn.close()

## 8. Run Each Concurrency Test

Defining the functions that call pessimistic and optimistic workers for the defined number of threads.

In [9]:
def run_pessimistic():
    threads = []
    start_time = time.time()

    for i in range(NUM_THREADS):
        t = threading.Thread(
            target=pessimistic_worker,
            args=(i,)
        )
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    return round(time.time() - start_time, 2)


def run_optimistic():
    threads = []
    start_time = time.time()

    for i in range(NUM_THREADS):
        t = threading.Thread(
            target=optimistic_worker,
            args=(i,)
        )
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    return round(time.time() - start_time, 2)

## 9. Read MySQL Log Counts

Defines the function that calls metrics from the MySQL database to show number of conflicts, deadlocks, and total transactions.

In [10]:
def get_db_metrics(strategy):
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("SELECT COUNT(*) FROM conflict_log")
    conflict_count = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM deadlock_log")
    deadlock_count = cursor.fetchone()[0]

    cursor.execute("""
        SELECT COUNT(*)
        FROM transaction_log
        WHERE strategy = %s
    """, (strategy,))
    transaction_count = cursor.fetchone()[0]

    cursor.close()
    conn.close()

    return conflict_count, deadlock_count, transaction_count


## 10. Run the Full Test

Each strategy starts from a fresh test environment. Metrics are captured immediately after each test so the next reset does not erase them.

In [11]:
def run_scenario(scenario_name):
    global metrics
    # Reset metrics dict
    for strategy in metrics:
        for key in metrics[strategy]:
            metrics[strategy][key] = 0

    print(f"\n--- RUNNING SCENARIO: {scenario_name.upper()} ---")

    # 1. Pessimistic Test
    print("Resetting environment for pessimistic test...")
    reset_test_environment()  # Removed scenario_name argument
    
    threads = []
    start_time = time.time()
    for i in range(NUM_THREADS):
        t = threading.Thread(target=pessimistic_worker, args=(i, scenario_name))
        threads.append(t)
        t.start()
    for t in threads:
        t.join()
    pess_time = round(time.time() - start_time, 2)
    pess_conflicts, pess_deadlocks, pess_transactions = get_db_metrics("pessimistic")

    # 2. Optimistic Test
    print("Resetting environment for optimistic test...")
    reset_test_environment()  # Removed scenario_name argument

    threads = []
    start_time = time.time()
    for i in range(NUM_THREADS):
        t = threading.Thread(target=optimistic_worker, args=(i, scenario_name))
        threads.append(t)
        t.start()
    for t in threads:
        t.join()
    opt_time = round(time.time() - start_time, 2)
    opt_conflicts, opt_deadlocks, opt_transactions = get_db_metrics("optimistic")

    # Print Results Table
    print(f"\n=====================================================================================")
    print(f"RESULTS: {scenario_name.upper()} SCENARIO")
    print(f"=====================================================================================")
    print(f"{'Metric':<25}{'Pessimistic':<20}{'Optimistic':<20}")
    print(f"-------------------------------------------------------------------------------------")
    print(f"{'Runtime (seconds)':<25}{pess_time:<20}{opt_time:<20}")
    print(f"{'Success Count':<25}{metrics['pessimistic']['success']:<20}{metrics['optimistic']['success']:<20}")
    print(f"{'Exception Count':<25}{metrics['pessimistic']['errors']:<20}{metrics['optimistic']['errors']:<20}")
    print(f"{'Deadlocks (SQL Error: 1213)':<25}{metrics['pessimistic']['deadlocks']:<20}{metrics['optimistic']['deadlocks']:<20}")
    print(f"{'Timeouts (SQL Error: 1205)':<25}{metrics['pessimistic']['timeouts']:<20}{metrics['optimistic']['timeouts']:<20}")
    print(f"-------------------------------------------------------------------------------------")
    print(f"{'Conflict Log Rows':<25}{pess_conflicts:<20}{opt_conflicts:<20}")
    print(f"{'Deadlock Log Rows':<25}{pess_deadlocks:<20}{opt_deadlocks:<20}")
    print(f"{'Transaction Log Rows':<25}{pess_transactions:<20}{opt_transactions:<20}")
    print(f"=====================================================================================\n")

# Execute both scenarios
run_scenario("high_contention")
run_scenario("low_contention")


--- RUNNING SCENARIO: HIGH_CONTENTION ---
Resetting environment for pessimistic test...


C:\Users\proza\AppData\Local\Temp\ipykernel_4048\3502692008.py:15: DeprecationWarning: Call to deprecated function stored_results. Reason: The property counterpart 'stored_results' will be added in a future release, and this method will be removed.
  for result in cursor.stored_results():


Resetting environment for optimistic test...


C:\Users\proza\AppData\Local\Temp\ipykernel_4048\1416775093.py:18: DeprecationWarning: Call to deprecated function stored_results. Reason: The property counterpart 'stored_results' will be added in a future release, and this method will be removed.
  for result in cursor.stored_results():



RESULTS: HIGH_CONTENTION SCENARIO
Metric                   Pessimistic         Optimistic          
-------------------------------------------------------------------------------------
Runtime (seconds)        35.37               121.39              
Success Count            24999               25000               
Exception Count          1                   1                   
Deadlocks (SQL Error: 1213)0                   0                   
Timeouts (SQL Error: 1205)0                   0                   
-------------------------------------------------------------------------------------
Conflict Log Rows        0                   22769               
Deadlock Log Rows        0                   0                   
Transaction Log Rows     24999               25000               


--- RUNNING SCENARIO: LOW_CONTENTION ---
Resetting environment for pessimistic test...
Resetting environment for optimistic test...

RESULTS: LOW_CONTENTION SCENARIO
Metric                   Pes

## 11. Results and Analysis

The measurements below compare the two locking strategies under two workloads. Each run used 50 threads and 500 operations per thread (25,000 operations per strategy). The scenarios differ in the number of available copies and the proportion of writes, so the results show how workload shape affects the better choice.

### Summary comparison

| Scenario | Workload | Pessimistic runtime | Optimistic runtime | Exceptions / timeouts | Conflict log rows | Faster strategy |
|---|---|---:|---:|---:|---:|---|
| High contention | 2 shared copies; 100% checkout/write operations | 35.37 s | 121.39 s | Not reported here | 22,769 (optimistic) | Pessimistic (~3.4× faster) |
| Low contention | 15,000 copies; 98% reads / 2% writes; optimistic read-only path skips `conn.commit()` | 8.14 s | 9.24 s | 0 for both | Not reported here | Optimistic (~1.14× faster) |

### High contention: write-heavy hotspot

With only two copies available, the workload concentrates writes on the same rows. Optimistic locking must detect a version mismatch after a conflicting update, roll back, and retry. The 22,769 conflict log rows are evidence of substantial retry activity; repeated failed attempts add database and CPU work, creating a retry storm. In this case, the optimistic run took 121.39 seconds, while the pessimistic run completed in 35.37 seconds—about 3.4 times faster.

Pessimistic locking queues competing updates at the row level through `SELECT ... FOR UPDATE`. That serializes access to each hot row, but avoids repeated speculative updates and retries. For this highly contended workload, orderly waiting was substantially cheaper than optimistic conflict recovery.

### Low contention: distributed, read-heavy workload

The 15,000-copy pool spreads operations across many rows, and only 2% of operations write. With few write conflicts, optimistic locking can use non-blocking snapshot reads. Its read-only path also skips the transaction commit round-trip (`conn.commit()`), reducing overhead for the dominant read workload. Optimistic locking completed in 8.14 seconds compared with 9.24 seconds for pessimistic locking, with zero exceptions or timeouts for either strategy.

Under these conditions, avoiding row-lock acquisition and unnecessary read-only commits outweighed the optimistic strategy's conflict-checking costs. The observed runtime advantage was modest (about 1.14×), and applies to this workload and implementation.

### Executive takeaway

There is no universally faster locking strategy in these tests. Pessimistic locking is the better fit for concentrated, write-heavy hotspots because it limits wasted work from repeated conflicts. Optimistic locking is the better fit for widely distributed, read-heavy traffic when writes are uncommon and read-only transactions avoid unnecessary commits. Choose based on expected contention and write frequency, and validate with representative workloads before drawing broader performance conclusions.
